# Credit Card Fraud Detection - Model Deployment

Production deployment pipeline: creating serializable pipelines, exporting `.pkl` files, constructing a real-time transaction prediction function, and latency benchmarking.

In [ ]:
# Step 1: Import Libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib
import time
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Step 2: Load the Dataset
try:
    data = pd.read_csv("../creditcard.csv")
    print("✅ Loaded ../creditcard.csv")
except FileNotFoundError:
    data = pd.read_csv("creditcard.csv")
    print("✅ Loaded creditcard.csv")

# Separate features and labels
X = data.drop(columns="Class", axis=1)
y = data["Class"]
print(f"Loaded {X.shape[0]:,} transactions with {X.shape[1]} features.")

In [ ]:
# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=2
)
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
# Step 4: Create Pipeline and Train Model
model_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("log_reg", LogisticRegression(solver="lbfgs", class_weight='balanced', max_iter=1000, random_state=2))
])

print("Fitting production pipeline...")
model_pipeline.fit(X_train, y_train)
print("Pipeline trained successfully!")

In [ ]:
# Step 5: Save Serialized Artifacts
# Save both to local and parent folder if running from Code/
model_path = "fraud_detection_model.pkl"
scaler_path = "scaler.pkl"

joblib.dump(model_pipeline, model_path)
joblib.dump(model_pipeline.named_steps['scaler'], scaler_path)

print(f"Saved: {model_path} ({os.path.getsize(model_path):,} bytes)")
print(f"Saved: {scaler_path} ({os.path.getsize(scaler_path):,} bytes)")

In [ ]:
# Step 6: Verify Loading from Disk
loaded_pipeline = joblib.load(model_path)
test_accuracy = loaded_pipeline.score(X_test, y_test)
print(f"Loaded pipeline test score: {test_accuracy:.4f} ✅")

## Real-Time Transaction Prediction Function

In [ ]:
def predict_transaction(transaction_dict, threshold=0.5):
    '''
    Evaluates a single incoming transaction in real-time.
    
    Parameters:
        transaction_dict (dict or pd.Series): Transaction feature values
        threshold (float): Fraud decision boundary (default 0.5)
        
    Returns:
        dict: Fraud prediction, risk score, and recommended action
    '''
    df_tx = pd.DataFrame([transaction_dict])
    
    # Ensure correct feature alignment
    for col in X.columns:
        if col not in df_tx.columns:
            df_tx[col] = 0.0
    df_tx = df_tx[X.columns]
    
    fraud_prob = loaded_pipeline.predict_proba(df_tx)[0, 1]
    is_fraud = bool(fraud_prob >= threshold)
    
    if fraud_prob < 0.2:
        risk_level = "LOW RISK - APPROVE"
    elif fraud_prob < 0.6:
        risk_level = "MEDIUM RISK - 2FA CHALLENGE"
    else:
        risk_level = "CRITICAL RISK - BLOCK TRANSACTION"
        
    return {
        "is_fraud": is_fraud,
        "fraud_probability": round(float(fraud_prob), 4),
        "risk_level": risk_level,
        "decision": "DECLINE" if is_fraud else "APPROVE"
    }

## Live Inference Testing on Real Test Samples

In [ ]:
# Sample 1: A legitimate transaction
normal_sample = X_test[y_test == 0].iloc[10].to_dict()
result_normal = predict_transaction(normal_sample)

print("🧪 Test Case 1: Legitimate Transaction")
for k, v in result_normal.items():
    print(f"  {k}: {v}")

# Sample 2: A fraudulent transaction
fraud_sample = X_test[y_test == 1].iloc[5].to_dict()
result_fraud = predict_transaction(fraud_sample)

print("\n🧪 Test Case 2: Fraudulent Transaction")
for k, v in result_fraud.items():
    print(f"  {k}: {v}")

## Latency and Throughput Benchmarking

In [ ]:
# Measure single transaction latency
iterations = 1000
test_record = normal_sample

start_time = time.perf_counter()
for _ in range(iterations):
    _ = predict_transaction(test_record)
elapsed = time.perf_counter() - start_time

avg_latency_ms = (elapsed / iterations) * 1000
throughput = iterations / elapsed

print(f"Latency per Transaction: {avg_latency_ms:.3f} ms")
print(f"Throughput:             {throughput:,.0f} transactions/second")
print("Performance is well within the typical 50ms payment gateway SLA!")

## Production REST API Example (FastAPI Blueprint)

```python
from fastapi import FastAPI
from pydantic import BaseModel
import joblib

app = FastAPI(title="Credit Card Fraud Detection API", version="1.0")
model = joblib.load("fraud_detection_model.pkl")

class Transaction(BaseModel):
    Time: float
    Amount: float
    V1: float
    V2: float
    # ... V3 to V28

@app.post("/predict")
def predict_fraud(tx: Transaction):
    data = [list(tx.dict().values())]
    prob = float(model.predict_proba(data)[0, 1])
    return {
        "fraud_probability": prob,
        "is_fraud": prob >= 0.5,
        "status": "DECLINED" if prob >= 0.5 else "APPROVED"
    }
```
